# Phase 7: Polish & Presentation

This notebook demonstrates:
1. **Rich formatters** — Markdown tables, cards, and charts for travel plan output
2. **Budget visualization** — matplotlib pie chart of spending breakdown
3. **Packing suggestions** — weather-based recommendations
4. **Streaming** — watch agents complete in real-time with `graph.stream()`
5. **Error handling** — graceful degradation when data is missing

The Gradio web app (`src/app.py`) uses these same formatters for a standalone demo.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

from langchain_core.messages import HumanMessage
from langgraph.types import Command
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

from src.graph_builder import build_graph
from src.utils.formatters import (
    format_full_plan, format_trip_summary_card, format_flight_table,
    format_hotel_table, format_itinerary, format_budget_text,
    create_budget_pie_chart, format_packing_suggestions,
)

print("All imports successful!")

## 2. Run a Trip & Display Rich Formatters

Build the graph with human-in-the-loop and run a Maldives trip.

In [ ]:
graph = build_graph(include_human_review=True)
config = {"configurable": {"thread_id": "notebook-07-demo"}}

print("Running: 7-day Maldives trip...")
result = graph.invoke(
    {"messages": [HumanMessage(content=(
        "Plan a 7-day luxury beach vacation in the Maldives for 2 people, "
        "budget $5000, departing from New York. "
        "Dates: 2025-06-10 to 2025-06-17. "
        "Preferences: beach, snorkeling, spa."
    ))]},
    config,
)

state = graph.get_state(config)
print(f"Plan status: {state.values.get('plan_status')}")
print(f"All agents completed: {all(state.values.get(f) is not None for f in ['destination_research', 'flight_options', 'hotel_options', 'itinerary', 'budget_summary'])}")

### Trip Summary Card

In [ ]:
display(Markdown(format_trip_summary_card(
    state.values.get("travel_request", {}),
    state.values.get("destination_research", {}),
)))

### Flight Comparison Table

In [ ]:
display(Markdown(format_flight_table(state.values.get("flight_options", {}))))

### Hotel Comparison Table

In [ ]:
display(Markdown(format_hotel_table(state.values.get("hotel_options", {}))))

### Day-by-Day Itinerary

In [ ]:
display(Markdown(format_itinerary(state.values.get("itinerary", {}))))

### Budget Breakdown

In [ ]:
display(Markdown(format_budget_text(state.values.get("budget_summary", {}))))

## 3. Budget Pie Chart

In [ ]:
fig = create_budget_pie_chart(state.values.get("budget_summary", {}))
if fig:
    plt.show()
else:
    print("No budget data available for chart.")

## 4. Packing Suggestions

In [ ]:
display(Markdown(format_packing_suggestions(
    state.values.get("itinerary", {}),
    state.values.get("destination_research", {}),
)))

## 5. Streaming Demo

Use `graph.stream()` with `stream_mode="updates"` to see agent-by-agent progress in real-time.

In [ ]:
# Non-HIL graph for clean streaming (no interrupt pause)
stream_graph = build_graph(include_human_review=False)

print("Streaming agent-by-agent updates:")
print()

node_labels = {
    "intent_parser": "Parsing request",
    "destination_research": "Researching destination",
    "flight_search": "Searching flights",
    "hotel_search": "Searching hotels",
    "activity_itinerary": "Planning activities",
    "budget_currency": "Analyzing budget",
    "aggregator": "Assembling plan",
}

for step in stream_graph.stream(
    {"messages": [HumanMessage(content=(
        "Plan a 5-day cultural trip to Paris for 2 people, "
        "budget $4000, from Chicago. Dates: 2025-04-01 to 2025-04-06. "
        "Preferences: culture, food, history."
    ))]},
    stream_mode="updates",
):
    node_name = list(step.keys())[0]
    label = node_labels.get(node_name, node_name)
    # Extract first message content for a short status
    node_data = step[node_name]
    msg = ""
    if "messages" in node_data:
        for m in node_data["messages"]:
            if hasattr(m, "content"):
                msg = m.content[:100]
                break
    print(f">> [{label}] {msg}")

print()
print("Streaming complete!")

## 6. Error Handling — Graceful Degradation

All formatters handle `None` or empty data without crashing.

In [ ]:
print("Formatters with None/empty inputs:")
print()

display(Markdown("**Flight table (None):**"))
display(Markdown(format_flight_table(None)))

display(Markdown("**Hotel table (empty dict):**"))
display(Markdown(format_hotel_table({})))

display(Markdown("**Itinerary (empty days):**"))
display(Markdown(format_itinerary({"days": []})))

display(Markdown("**Budget (None):**"))
display(Markdown(format_budget_text(None)))

display(Markdown("**Packing (None):**"))
display(Markdown(format_packing_suggestions(None, None)))

display(Markdown("**Full plan (empty state):**"))
empty_plan = format_full_plan({})
for section, content in empty_plan.items():
    display(Markdown(f"*{section}*: {content[:60]}..."))

print()
print("All formatters handled gracefully — no crashes!")

## 7. Full Formatted Plan

Approve the Maldives plan and display all sections via `format_full_plan()`.

In [ ]:
# Approve the plan from section 2
result = graph.invoke(Command(resume="approve"), config)
state = graph.get_state(config)
print(f"Plan status: {state.values.get('plan_status')}")

plan = format_full_plan(state.values)
for section_name, content in plan.items():
    display(Markdown(f"---\n## {section_name.replace('_', ' ').title()}"))
    display(Markdown(content))

## 8. Verification

In [ ]:
checks = [
    ("Formatters produce non-empty output", all(v and len(v) > 10 for v in plan.values())),
    ("Budget chart is a Figure", fig is not None),
    ("Flight table has markdown table syntax", "|" in plan["flights"]),
    ("Hotel table has markdown table syntax", "|" in plan["hotels"]),
    ("Itinerary has day entries", "Day" in plan["itinerary"]),
    ("Budget has category breakdown", "Flights" in plan["budget"]),
    ("Packing has suggestions", len(plan["packing"]) > 50),
    ("Summary has destination name", "Maldives" in plan["summary"] or "maldives" in plan["summary"].lower()),
    ("Graceful degradation works", all(isinstance(v, str) for v in format_full_plan({}).values())),
    ("Graph builder (HIL) has human_review", "human_review" in [n for n in graph.get_graph().nodes]),
]

pass_count = sum(1 for _, r in checks if r)
for name, result in checks:
    status = "PASS" if result else "FAIL"
    print(f"  [{status}] {name}")

print(f"\n  {pass_count}/{len(checks)} checks passed")

## Summary

### What we demonstrated:
1. **Rich formatters** — Markdown tables for flights, hotels, itinerary; trip summary card; budget breakdown
2. **Budget visualization** — matplotlib pie chart with percentage labels
3. **Packing suggestions** — weather-aware recommendations based on itinerary data
4. **Streaming** — `graph.stream(stream_mode="updates")` shows agent-by-agent progress
5. **Error handling** — all formatters degrade gracefully with missing data
6. **Graph builder** — `build_graph()` centralizes graph construction

### Gradio Web App
Run the interactive web UI:
```bash
uv run python -m src.app
# Open http://localhost:7860
```